In [1]:
# 🔹 1. RELOAD AUTOMÁTICO
# POR QUÉ: Jupyter cachea imports. Si modificas loader.py y no reinicias el kernel,
# sigue usando la versión vieja. autoreload 2 evita reinicios manuales y garantiza
# que ejecutas siempre el código actualizado.
%load_ext autoreload
%autoreload 2

# 🔹 2. RESOLUCIÓN DE RUTA RAÍZ
# POR QUÉ: El notebook corre en notebooks/. Python resuelve rutas desde ahí.
# Esta línea busca la carpeta config/ en el CWD actual; si no está, sube un nivel.
# Garantiza que PROJECT_ROOT siempre apunte a Concurso_Data/, sin importar dónde abras el .ipynb.
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd() if (Path.cwd() / "config").exists() else Path.cwd().parent

# 🔹 3. INYECCIÓN DE sys.path
# POR QUÉ: Python no busca automáticamente en src/. Sin esto, from loader import DataLoader
# falla con ModuleNotFoundError cuando el CWD es notebooks/. Se inyecta solo para esta sesión.
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# 🔹 4. INSTANCIA DEL LOADER
# POR QUÉ: Pasamos la ruta absoluta del JSON para que el DataLoader no adivine rutas relativas.
from loader import DataLoader
CONFIG_PATH = str(PROJECT_ROOT / "config" / "mapping_config.json")
loader = DataLoader(config_path=CONFIG_PATH)

# 🔹 5. CARGA DE LOS 4 DATASETS
# POR QUÉ: Tu Etapa 1 exige cargar los 4 datasets. DS4 debe validarse estructuralmente ahora
# (columnas clave, existencia de archivo). Si falla, el pipeline se detiene antes de consumir RAM innecesaria.
raw = loader.load_all_datasets(['violencia', 'sexuales', 'poblacion', 'forense', 'seforense'])

# 🔹 6. VERIFICACIÓN DE SALIDA
# POR QUÉ: Confirma que la Etapa 1 cerró correctamente: 4 DataFrames en RAM, dimensiones esperadas,
# columnas clave presentes. Si alguna falla, el log del loader ya indicó el motivo exacto.
print("\n📊 RESULTADOS ETAPA 1:")
for k, v in raw.items():
    print(f"{k.capitalize():<12}: {v.shape} | Primeras cols: {v.columns[:15].tolist()}")

2026-06-24 10:36:40,806 - INFO - 📁 DataLoader inicializado en: /Users/anaaguirre/Documents/Cicatrices_invisibles/data
2026-06-24 10:36:40,806 - INFO - 🔍 Anclado a raíz de proyecto: /Users/anaaguirre/Documents/Cicatrices_invisibles
2026-06-24 10:36:40,806 - INFO - 🚀 Cargando 5 dataset(s)...
2026-06-24 10:36:41,187 - INFO - ✅ violencia cargado: 682,558 filas, 8 cols
2026-06-24 10:36:41,484 - INFO - ✅ sexuales cargado: 392,576 filas, 9 cols
2026-06-24 10:37:55,432 - INFO - ✅ poblacion cargado: 84,233 filas, 312 cols
2026-06-24 10:37:56,832 - INFO - ✅ forense cargado: 236,840 filas, 36 cols
2026-06-24 10:37:57,660 - INFO - ✅ seforense cargado: 233,352 filas, 32 cols
2026-06-24 10:37:57,660 - INFO - ✅ Carga completada: 5/5 exitosos.



📊 RESULTADOS ETAPA 1:
Violencia   : (682558, 8) | Primeras cols: ['DEPARTAMENTO', 'MUNICIPIO', 'CODIGO DANE', 'ARMAS MEDIOS', 'FECHA HECHO', 'GENERO', 'GRUPO ETARIO', 'CANTIDAD']
Sexuales    : (392576, 9) | Primeras cols: ['DEPARTAMENTO', 'MUNICIPIO', 'CODIGO DANE', 'ARMAS MEDIOS', 'FECHA HECHO', 'GENERO', 'GRUPO ETARIO', 'CANTIDAD', 'delito']
Poblacion   : (84233, 312) | Primeras cols: ['DP', 'DPNOM', 'MPIO', 'DPMP', 'AÑO', 'ÁREA GEOGRÁFICA', 'TOTAL Total', 'TOTAL Hombres', 'TOTAL Mujeres', 'HOMBRES Hombres 0 años', 'HOMBRES Hombres 1 año', 'HOMBRES Hombres 2 años', 'HOMBRES Hombres 3 años', 'HOMBRES Hombres 4 años', 'HOMBRES Hombres 5 años']
Forense     : (236840, 36) | Primeras cols: ['ID', 'Año del hecho', 'Sexo de la victima', 'Grupo de edad quinquenal', 'Grupo Mayor Menor de Edad', 'Grupo de Edad judicial', 'Ciclo Vital', 'País de Nacimiento', 'Escolaridad', 'Estado Civil', 'Tipo de Discapacidad', 'Pertenencia Étnica', 'Orientación Sexual', 'Identidad de Género', 'Transgénero']


In [2]:
import sys
from pathlib import Path

%load_ext autoreload
%autoreload 2

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "config").exists() else Path.cwd().parent
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

#from loader import DataLoader
from metadata_mapper import MetadataMapper

# 1. Carga cruda
#loader = DataLoader(config_path=str(PROJECT_ROOT / "config" / "mapping_config.json"))
#raw = loader.load_all_datasets(['violencia', 'sexuales', 'poblacion', 'forense'])

# 2. Rutas reales
raw_paths = {
    k: loader.base_path / loader.paths.get('raw_dir', 'raw') / loader.datasets_config[k]['filename']
    for k in raw.keys()
}

# 3. Mapeo estructural (ahora blindado)
mapper = MetadataMapper(config_path=str(PROJECT_ROOT / "config" / "mapping_config.json"))
reporte = mapper.map_all(raw, raw_paths=raw_paths)

# 4. Visualización
mapper.print_tablero(reporte)

print("\n✅ Etapa 2 cerrada: MetadataMapper ejecutado sin errores.")

2026-06-24 10:40:02,799 - INFO - 📂 Ruta de docs validada: /Users/anaaguirre/Documents/Cicatrices_invisibles/docs
2026-06-24 10:40:02,799 - INFO - 🚀 Mapeando 5 dataset(s)...
2026-06-24 10:40:02,800 - INFO - 📸 Mapeando estructura: violencia
2026-06-24 10:40:02,806 - INFO - 📸 Mapeando estructura: sexuales
2026-06-24 10:40:02,808 - INFO - 📸 Mapeando estructura: poblacion
2026-06-24 10:40:02,977 - INFO - 📸 Mapeando estructura: forense
2026-06-24 10:40:02,981 - INFO - 📸 Mapeando estructura: seforense
2026-06-24 10:40:02,989 - INFO - 📄 Reporte exportado: /Users/anaaguirre/Documents/Cicatrices_invisibles/docs/mapeo_estructural_raw.csv


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload

📊 TABLERO DE RECONOCIMIENTO ESTRUCTURAL

📦 VIOLENCIA
   📐 Estructura: 682,558 filas × 8 columnas
   💾 Memoria RAM: 82.33 MB | Disco: 60.61 MB
   🧬 Tipos: {'DEPARTAMENTO': 'string', 'MUNICIPIO': 'string', 'CODIGO DANE': 'Int64', 'ARMAS MEDIOS': 'string', 'FECHA HECHO': 'string', 'GENERO': 'string', 'GRUPO ETARIO': 'string', 'CANTIDAD': 'Int64'}
   📋 Schema: ['DEPARTAMENTO', 'MUNICIPIO', 'CODIGO DANE', 'ARMAS MEDIOS', 'FECHA HECHO']...

📦 SEXUALES
   📐 Estructura: 392,576 filas × 9 columnas
   💾 Memoria RAM: 69.61 MB | Disco: 55.24 MB
   🧬 Tipos: {'DEPARTAMENTO': 'string', 'MUNICIPIO': 'string', 'CODIGO DANE': 'Int64', 'ARMAS MEDIOS': 'string', 'FECHA HECHO': 'string', 'GENERO': 'string', 'GRUPO ETARIO': 'string', 'CANTIDAD': 'Int64', 'delito': 'string'}
   📋 Schema: ['DEPARTAMENTO', 'MUNICIPIO', 'CODIGO DANE', 'ARMAS MEDIOS', 'FECHA HECHO']...

📦 POBLACION
   📐 Estructura: 84,233 filas × 312 columna

In [3]:
import sys
from pathlib import Path

%load_ext autoreload
%autoreload 2

# 🔒 Seguridad: verifica que 'raw' exista en memoria (viene de la Celda 1)
if 'raw' not in globals():
    raise RuntimeError("⚠️ Ejecuta primero la Celda 1 (DataLoader). 'raw' no está en memoria.")

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "config").exists() else Path.cwd().parent
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from data_standardizer import DataStandardizer

# 1. Ejecutar estandarización
print("🔧 Iniciando estandarización de los datasets...")
std = DataStandardizer(config_path=str(PROJECT_ROOT / "config" / "mapping_config.json"))
standardized = std.standardize_all(raw)

# 2. Validación detallada y defensiva
print("\n✅ CELDA 3 - DATA STANDARDIZER EJECUTADO")
print("="*60)

for name, df in standardized.items():
    print(f"\n📦 {name.upper()}")
    print(f"   Filas: {len(df):,} | Columnas: {len(df.columns)}")

    claves = [c for c in ['fecha_hecho', 'cod_municipio', 'departamento'] if c in df.columns]
    print(f"   📌 Claves críticas: {claves}")

    if 'fecha_hecho' in df.columns:
        dtype = df['fecha_hecho'].dtype
        nulos = df['fecha_hecho'].isna().sum()
        muestra = df['fecha_hecho'].dropna().head(2).tolist()
        print(f"   📅 fecha_hecho: {dtype} | NaT: {nulos} | Muestra: {muestra}")

    if 'departamento' in df.columns:
        unicos = [str(x) for x in df['departamento'].dropna().unique()[:5]]
        print(f"   📍 Departamentos (muestra): {unicos}")


    # 📊 DANE específico
    if name == 'poblacion':
        pob_cols = [c for c in df.columns if c.startswith('pob_')]
        print(f"   📊 Agregación DANE: {'✅ SÍ' if 'pob_f_0_17' in df.columns else '❌ NO'}")
        if pob_cols:
            print(f"      • Columnas: {pob_cols}")

print("\n🛡️ Verificación final:")
print(f"• Datasets procesados: {len(standardized)}/{len(loader.datasets_config)}")
print("✅ Etapa 3 completada. Listo para DataSegmenter.")

2026-06-24 10:40:08,499 - INFO - 🚀 Estandarizando 5 dataset(s)...
2026-06-24 10:40:08,499 - INFO - 🔧 Estandarizando: violencia (682,558 filas, 8 cols)


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
🔧 Iniciando estandarización de los datasets...


2026-06-24 10:40:10,418 - INFO - 🏷️ Geo-Integridad: 4 códigos inválidos → 'CODIGO_MAL_REGISTRADO'
2026-06-24 10:40:10,472 - INFO - ✅ violencia estandarizado. Columnas finales: 11
2026-06-24 10:40:10,472 - INFO - 🔧 Estandarizando: sexuales (392,576 filas, 9 cols)
2026-06-24 10:40:11,576 - INFO - 🏷️ Geo-Integridad: 3 códigos inválidos → 'CODIGO_MAL_REGISTRADO'
2026-06-24 10:40:11,961 - INFO - ✅ sexuales estandarizado. Columnas finales: 13
2026-06-24 10:40:11,961 - INFO - 🔧 Estandarizando: poblacion (84,233 filas, 312 cols)
2026-06-24 10:40:12,627 - INFO - ✅ poblacion estandarizado. Columnas finales: 115
2026-06-24 10:40:12,628 - INFO - 🔧 Estandarizando: forense (236,840 filas, 36 cols)
2026-06-24 10:40:16,104 - INFO - ✅ forense estandarizado. Columnas finales: 41
2026-06-24 10:40:16,104 - INFO - 🔧 Estandarizando: seforense (233,352 filas, 32 cols)
2026-06-24 10:40:19,260 - INFO - ✅ seforense estandarizado. Columnas finales: 38



✅ CELDA 3 - DATA STANDARDIZER EJECUTADO

📦 VIOLENCIA
   Filas: 682,558 | Columnas: 11
   📌 Claves críticas: ['fecha_hecho', 'cod_municipio', 'departamento']
   📅 fecha_hecho: datetime64[us] | NaT: 0 | Muestra: [Timestamp('2026-03-10 00:00:00'), Timestamp('2026-03-15 00:00:00')]
   📍 Departamentos (muestra): ['CALDAS', 'ATLANTICO', 'HUILA', 'CUNDINAMARCA', 'VALLE DEL CAUCA']

📦 SEXUALES
   Filas: 392,576 | Columnas: 13
   📌 Claves críticas: ['fecha_hecho', 'cod_municipio', 'departamento']
   📅 fecha_hecho: datetime64[us] | NaT: 0 | Muestra: [Timestamp('2010-02-04 00:00:00'), Timestamp('2010-02-23 00:00:00')]
   📍 Departamentos (muestra): ['PUTUMAYO', 'CASANARE', 'VALLE DEL CAUCA', 'CORDOBA', 'GUAJIRA']

📦 POBLACION
   Filas: 84,233 | Columnas: 115
   📌 Claves críticas: ['fecha_hecho', 'cod_municipio', 'departamento']
   📅 fecha_hecho: datetime64[ns] | NaT: 8 | Muestra: [Timestamp('2018-01-01 00:00:00'), Timestamp('2018-01-01 00:00:00')]
   📍 Departamentos (muestra): ['ANTIOQUIA', 'ATLA

In [4]:
import sys
import json
from pathlib import Path

%load_ext autoreload
%autoreload 2

if 'standardized' not in globals():
    raise RuntimeError("⚠️ Ejecuta primero la Etapa 3 (DataStandardizer).")

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "config").exists() else Path.cwd().parent
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# ✅ CAMBIO AQUÍ: Fusionar base_config.json + mapping_config.json
base_path = PROJECT_ROOT / "config" / "base_config.json"
base_cfg = {}
if base_path.exists():
    with open(base_path, 'r', encoding='utf-8') as f:
        base_cfg = json.load(f)
with open(PROJECT_ROOT / "config" / "mapping_config.json", 'r', encoding='utf-8') as f:
    spec_cfg = json.load(f)
config = {**base_cfg, **spec_cfg}  # El específico gana prioridad

# ✅ Ahora sí encuentra display_config (vive en base_config.json)
disp = config.get('display_config')
if not disp: raise ValueError("❌ 'display_config' no definido en la configuración fusionada")

SEP = disp['separator_char'] * disp['separator_length']
IND = disp['indent']
PREFIX_DS = disp['prefix_dataset']
TITLE = disp['section_title'].format(n=4, module="DATA SEGMENTER")

from data_segmenter import DataSegmenter

print(f"✂️ SEGMENTACIÓN VECTORIAL (Departamento Y Año 2018-2025)")
print(SEP)

segmenter = DataSegmenter(config_path=str(PROJECT_ROOT / "config" / "mapping_config.json"))
metrics_df = segmenter.segment_all(standardized)

print(f"\n{TITLE}")
print(SEP)

for _, row in metrics_df.iterrows():
    print(f"\n{PREFIX_DS}{row['dataset'].upper()}")
    print(f"{IND}📥 Originales: {row['filas_originales']:,}")
    print(f"{IND}✅ Cumplen AMBOS (Depto + Año): {row['filas_cumplen_ambos']:,}")
    print(f"{IND}⚠️  Excluidas (fuera de alcance): {row['excluidas_no_cumplen']:,}")
    print(f"{IND}📊 Columnas: {row['columnas_preservadas']} (100% conservadas)")
    print(f"{IND}💾 Guardado en: data/segmented/{Path(row['ruta_parquet']).name}")

print(f"\n🛡️ Garantías aplicadas:")
print(f"{IND}• Filtrado 100% vectorial: mask = df['depto'].isin() & df['año'].between()")
print(f"{IND}• Lógica AND estricta: solo entran filas con departamento válido Y año 2018-2025.")
print(f"{IND}• NaT en fecha_hecho → excluido nativamente (no cumple rango temporal).")
print(f"{IND}• Rutas dinámicas: combina base_data_dir + segmented_dir desde JSON. Cero carpetas huérfanas.")

print(SEP)
print(f"✅ Etapa 4 completada. Listo para DataAuditor.")

2026-06-24 10:40:28,684 - INFO - 📂 Usando carpeta configurada: /Users/anaaguirre/Documents/Cicatrices_invisibles/data/segmented
2026-06-24 10:40:28,684 - INFO - ✂️ Segmentando (AND Vectorial): Depto + Año 2018-2025


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
✂️ SEGMENTACIÓN VECTORIAL (Departamento Y Año 2018-2025)

✅ CELDA 4 - DATA SEGMENTER EJECUTADO

📦 VIOLENCIA
   📥 Originales: 682,558
   ✅ Cumplen AMBOS (Depto + Año): 68,073
   ⚠️  Excluidas (fuera de alcance): 614,485
   📊 Columnas: 11 (100% conservadas)
   💾 Guardado en: data/segmented/violencia_2018_2025.parquet

📦 SEXUALES
   📥 Originales: 392,576
   ✅ Cumplen AMBOS (Depto + Año): 37,802
   ⚠️  Excluidas (fuera de alcance): 354,774
   📊 Columnas: 13 (100% conservadas)
   💾 Guardado en: data/segmented/sexuales_2018_2025.parquet

📦 POBLACION
   📥 Originales: 84,233
   ✅ Cumplen AMBOS (Depto + Año): 4,296
   ⚠️  Excluidas (fuera de alcance): 79,937
   📊 Columnas: 115 (100% conservadas)
   💾 Guardado en: data/segmented/poblacion_2018_2025.parquet

📦 FORENSE
   📥 Originales: 236,840
   ✅ Cumplen AMBOS (Depto + Año): 15,418
   ⚠️  Excluidas (fuera de alcance): 221,422
   📊 Columnas: 41 (100% conservad

In [5]:
import sys
import json
from pathlib import Path
import pandas as pd

%load_ext autoreload
%autoreload 2

# 🔒 Anclaje determinista a raíz del proyecto
PROJECT_ROOT = Path.cwd() if (Path.cwd() / "config").exists() else Path.cwd().parent
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# ✅ CAMBIO AQUÍ: Fusión base + mapping para recuperar 'paths' y 'display_config'
base_path = PROJECT_ROOT / "config" / "base_config.json"
base_cfg = {}
if base_path.exists():
    with open(base_path, 'r', encoding='utf-8') as f: base_cfg = json.load(f)
with open(PROJECT_ROOT / "config" / "mapping_config.json", 'r', encoding='utf-8') as f:
    spec_cfg = json.load(f)
config = {**base_cfg, **spec_cfg}  # paths y display_config se inyectan desde base

# ✅ Rutas resueltas ABSOLUTAMENTE desde JSON + project_root
SEGMENTED_DIR = (PROJECT_ROOT / config['paths']['segmented_dir']).resolve()
DOCS_DIR = (PROJECT_ROOT / config['paths']['docs_dir']).resolve()

if not SEGMENTED_DIR.exists():
    raise RuntimeError(f"⚠️ Carpeta segmentada no encontrada: {SEGMENTED_DIR}. Ejecuta Etapa 4.")
if not DOCS_DIR.exists():
    raise FileNotFoundError(f"❌ Carpeta docs no encontrada: {DOCS_DIR}. Verifica estructura.")

disp = config.get('display_config')
if not disp: raise ValueError("❌ 'display_config' no definido en JSON")

SEP = disp['separator_char'] * disp['separator_length']
IND = disp['indent']
PREFIX_DS = disp['prefix_dataset']
TITLE = disp['section_title'].format(n=5, module="DATA AUDITOR")

# ✅ Importación exacta (archivo debe estar en src/data_auditor.py)
from data_auditor import DataAuditor

print(f"🔍 AUDITORÍA INTEGRAL (100% Columnas | 100% Filas)")
print(SEP)

segmented_data = {f.stem.split('_')[0]: pd.read_parquet(f) for f in SEGMENTED_DIR.glob("*.parquet")}
auditor = DataAuditor(config_path=str(PROJECT_ROOT / "config" / "mapping_config.json"))
summary_df = auditor.audit_all(segmented_data)

print(f"\n{TITLE}")
print(SEP)

for _, row in summary_df.iterrows():
    print(f"\n{PREFIX_DS}{row['dataset'].upper()}")
    print(f"{IND}📊 Filas auditadas: {row['rows']:,}")
    print(f"{IND}⚠️  Columnas con nulos: {row['cols_with_nulls']}")
    print(f"{IND}🪞 Duplicados espejo: {row['mirror_duplicates']:,}")
    print(f"{IND}🧬 Schema issues: {row['schema_issues']}")
    print(f"{IND}🌐 Dominio issues: {row['domain_violations']}")
    print(f"{IND}📅 Temporal issues: {row['temporal_issues']}")
    print(f"{IND}🔢 Numeric issues: {row['numeric_issues']}")
    print(f"{IND}📄 Reporte: docs/{Path(row['report_path']).name}")

print(f"\n🛡️ Garantías:")
print(f"{IND}• Rutas ancladas a project_root: cero carpetas duplicadas")
print(f"{IND}• 100% vectorial + dominio por columna + .astype('string') seguro")
print(f"{IND}• Reportes guardados en tu docs/ existente")
print(SEP)
print(f"✅ Etapa 5 completada.")

2026-06-24 10:40:34,104 - INFO - 📂 Ruta de reporte validada: /Users/anaaguirre/Documents/Cicatrices_invisibles/docs
2026-06-24 10:40:34,105 - INFO - 🚀 Auditando 5 dataset(s) de forma integral...
2026-06-24 10:40:34,105 - INFO - 🔍 Auditando: violencia (68,073 filas)
2026-06-24 10:40:34,124 - INFO - 🔍 Auditando: forense (15,418 filas)
2026-06-24 10:40:34,139 - INFO - 🔍 Auditando: sexuales (37,802 filas)
2026-06-24 10:40:34,151 - INFO - 🔍 Auditando: poblacion (4,296 filas)
2026-06-24 10:40:34,171 - INFO - 🔍 Auditando: seforense (18,661 filas)


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
🔍 AUDITORÍA INTEGRAL (100% Columnas | 100% Filas)

✅ CELDA 5 - DATA AUDITOR EJECUTADO

📦 VIOLENCIA
   📊 Filas auditadas: 68,073
   ⚠️  Columnas con nulos: 2
   🪞 Duplicados espejo: 21
   🧬 Schema issues: 0
   🌐 Dominio issues: 0
   📅 Temporal issues: 0
   🔢 Numeric issues: 1
   📄 Reporte: docs/auditoria_violencia.csv

📦 FORENSE
   📊 Filas auditadas: 15,418
   ⚠️  Columnas con nulos: 0
   🪞 Duplicados espejo: 0
   🧬 Schema issues: 0
   🌐 Dominio issues: 0
   📅 Temporal issues: 0
   🔢 Numeric issues: 0
   📄 Reporte: docs/auditoria_forense.csv

📦 SEXUALES
   📊 Filas auditadas: 37,802
   ⚠️  Columnas con nulos: 2
   🪞 Duplicados espejo: 3,210
   🧬 Schema issues: 0
   🌐 Dominio issues: 0
   📅 Temporal issues: 0
   🔢 Numeric issues: 0
   📄 Reporte: docs/auditoria_sexuales.csv

📦 POBLACION
   📊 Filas auditadas: 4,296
   ⚠️  Columnas con nulos: 0
   🪞 Duplicados espejo: 0
   🧬 Schema issues: 0
   🌐 Dominio 

In [6]:
import sys
import json
from pathlib import Path
import pandas as pd

%load_ext autoreload
%autoreload 2

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "config").exists() else Path.cwd().parent
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path: sys.path.insert(0, str(SRC_DIR))

# ✅ CAMBIO: Fusión base + mapping para recuperar 'paths' y 'display_config'
base_path = PROJECT_ROOT / "config" / "base_config.json"
base_cfg = {}
if base_path.exists():
    with open(base_path, 'r', encoding='utf-8') as f: base_cfg = json.load(f)
with open(PROJECT_ROOT / "config" / "mapping_config.json", 'r', encoding='utf-8') as f:
    spec_cfg = json.load(f)
config = {**base_cfg, **spec_cfg}  # paths y display_config se inyectan desde base

disp = config.get('display_config')
if not disp: raise ValueError("❌ 'display_config' no definido")
SEP = disp['separator_char'] * disp['separator_length']
IND = disp['indent']
PREFIX_DS = disp['prefix_dataset']
TITLE = disp['section_title'].format(n=6, module="DATA CLEANER")

from data_cleaner import DataCleaner

print(f"🧼 LIMPIEZA QUIRÚRGICA (JSON-Driven | Vectorial)")
print(SEP)

# Cargar segmentados
SEGMENTED_DIR = (PROJECT_ROOT / config['paths']['segmented_dir']).resolve()
segmented_data = {f.stem.split('_')[0]: pd.read_parquet(f) for f in SEGMENTED_DIR.glob("*.parquet")}

cleaner = DataCleaner(config_path=str(PROJECT_ROOT / "config" / "mapping_config.json"))
metrics_df = cleaner.clean_all(segmented_data)

print(f"\n{TITLE}")
print(SEP)

for _, row in metrics_df.iterrows():
    print(f"\n{PREFIX_DS}{row['dataset'].upper()}")
    print(f"{IND}📥 Entrada: {row['filas_entrada']:,} filas")
    print(f"{IND}📤 Salida limpia: {row['filas_salida']:,} filas")
    print(f"{IND}🚫 Duplicados eliminados: {row['duplicados_eliminados']:,}")
    print(f"{IND}💊 Nulos imputados: {row['nulos_imputados']:,}")
    print(f"{IND}💾 Guardado: data/cleaned/{Path(row['ruta_limpio']).name}")

print(f"\n🛡️ Garantías:")
print(f"{IND}• Deduplicación espejo con cuarentena en docs/")
print(f"{IND}• Imputación vectorial SOLO en campos críticos (JSON)")
print(f"{IND}• Archivos originales en segmented/ intactos")
print(SEP)
print(f"✅ Etapa 6 completada. Datos listos para MasterBuilder.")

2026-06-24 10:40:38,341 - INFO - 🚀 Iniciando limpieza de 5 dataset(s)...
2026-06-24 10:40:38,341 - INFO - 🧼 Limpiando: violencia (68,073 filas)
2026-06-24 10:40:38,350 - INFO - 🚫 21 duplicados espejo → cuarentena: cuarentena_duplicados_violencia.csv
2026-06-24 10:40:38,373 - INFO - 🧼 Limpiando: forense (15,418 filas)
2026-06-24 10:40:38,391 - INFO - 🧼 Limpiando: sexuales (37,802 filas)
2026-06-24 10:40:38,405 - INFO - 🚫 3210 duplicados espejo → cuarentena: cuarentena_duplicados_sexuales.csv
2026-06-24 10:40:38,418 - INFO - 🧼 Limpiando: poblacion (4,296 filas)
2026-06-24 10:40:38,434 - INFO - 🧼 Limpiando: seforense (18,661 filas)


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
🧼 LIMPIEZA QUIRÚRGICA (JSON-Driven | Vectorial)

✅ CELDA 6 - DATA CLEANER EJECUTADO

📦 VIOLENCIA
   📥 Entrada: 68,073 filas
   📤 Salida limpia: 68,052 filas
   🚫 Duplicados eliminados: 21
   💊 Nulos imputados: 383
   💾 Guardado: data/cleaned/violencia_limpio.parquet

📦 FORENSE
   📥 Entrada: 15,418 filas
   📤 Salida limpia: 15,418 filas
   🚫 Duplicados eliminados: 0
   💊 Nulos imputados: 0
   💾 Guardado: data/cleaned/forense_limpio.parquet

📦 SEXUALES
   📥 Entrada: 37,802 filas
   📤 Salida limpia: 34,592 filas
   🚫 Duplicados eliminados: 3,210
   💊 Nulos imputados: 189
   💾 Guardado: data/cleaned/sexuales_limpio.parquet

📦 POBLACION
   📥 Entrada: 4,296 filas
   📤 Salida limpia: 4,296 filas
   🚫 Duplicados eliminados: 0
   💊 Nulos imputados: 0
   💾 Guardado: data/cleaned/poblacion_limpio.parquet

📦 SEFORENSE
   📥 Entrada: 18,661 filas
   📤 Salida limpia: 18,661 filas
   🚫 Duplicados eliminados: 0
   💊

In [7]:
import pandas as pd, json
from pathlib import Path

# 🔒 Misma lógica de resolución de rutas que usa el pipeline
PROJECT_ROOT = Path.cwd().parent if "notebooks" in str(Path.cwd()).lower() else Path.cwd()

# ✅ CAMBIO: Fusión base + mapping para recuperar 'paths'
base_path = PROJECT_ROOT / "config" / "base_config.json"
base_cfg = {}
if base_path.exists():
    with open(base_path, 'r', encoding='utf-8') as f: base_cfg = json.load(f)
with open(PROJECT_ROOT / "config" / "mapping_config.json", encoding='utf-8') as f:
    spec_cfg = json.load(f)
cfg = {**base_cfg, **spec_cfg}  # paths se inyecta desde base

CLEAN_DIR = (PROJECT_ROOT / cfg["paths"]["cleaned_dir"]).resolve()

print("🔍 VERIFICACIÓN DE COLUMNAS POST-CLEANER\n" + "="*60)
for ds in ["violencia", "sexuales", "forense", "seforense", "poblacion"]:
    fpath = CLEAN_DIR / f"{ds}_limpio.parquet"
    if fpath.exists():
        df = pd.read_parquet(fpath)
        print(f"📊 {ds.upper():<10}: {len(df):,} filas | {len(df.columns)} cols")
        print(f"   🔹 {df.columns.tolist()}\n")
    else:
        print(f"❌ {ds.upper():<10}: No encontrado en {fpath.name}\n")
print(f"✅ Etapa 7 completada.")

🔍 VERIFICACIÓN DE COLUMNAS POST-CLEANER
📊 VIOLENCIA : 68,052 filas | 10 cols
   🔹 ['departamento', 'municipio', 'cod_municipio', 'fecha_hecho', 'anio_hecho', 'mes_hecho', 'dia_hecho', 'genero_victima', 'grupo_etario', 'cantidad']

📊 SEXUALES  : 34,592 filas | 11 cols
   🔹 ['departamento', 'municipio', 'cod_municipio', 'fecha_hecho', 'anio_hecho', 'mes_hecho', 'dia_hecho', 'genero_victima', 'grupo_etario', 'cantidad', 'dimension_delito']

📊 FORENSE   : 15,418 filas | 21 cols
   🔹 ['anio_hecho', 'genero_victima', 'ciclo_vital', 'grupo_edad', 'cod_municipio', 'municipio', 'departamento', 'escenario', 'agresor', 'sexo_agresor', 'factor', 'discapacidad', 'etnia', 'hora_rango', 'mes_hecho', 'dia_hecho', 'dias_incapacidad', 'dimension_agresor', 'dimension_escenario', 'dimension_discapacidad', 'dimension_etnia']

📊 SEFORENSE : 18,661 filas | 21 cols
   🔹 ['anio_hecho', 'genero_victima', 'grupo_edad', 'ciclo_vital', 'discapacidad', 'etnia', 'mes_hecho', 'dia_hecho', 'hora_rango', 'cod_municipio

In [8]:
import sys, json
from pathlib import Path
%load_ext autoreload
%autoreload 2

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "config").exists() else Path.cwd().parent
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path: sys.path.insert(0, str(SRC_DIR))

# ✅ CAMBIO: Fusión base + master_builder_config.json
base_path = PROJECT_ROOT / "config" / "base_config.json"
base_cfg = {}
if base_path.exists():
    with open(base_path, 'r', encoding='utf-8') as f: base_cfg = json.load(f)
with open(PROJECT_ROOT / "config" / "master_builder_config.json", 'r', encoding='utf-8') as f:
    spec_cfg = json.load(f)
config = {**base_cfg, **spec_cfg}  # display_config viene de base, aggregation de master_builder

disp = config['display_config']  # ✅ Ahora se resuelve desde la fusión
SEP = disp['separator_char'] * disp['separator_length']
TITLE = disp['section_title'].format(n=6, module="DATA AGGREGATOR")

from data_aggregator import DataAggregator
# ✅ IMPORTANTE: Apuntar al JSON donde vive aggregation_config
agg = DataAggregator(config_path=str(PROJECT_ROOT / "config" / "master_builder_config.json"))
layers_ram = agg.aggregate_all()

print(f"\n{TITLE}\n{SEP}")
for name, df in layers_ram.items():
    print(f"🔹 {name}: {len(df):,} filas")
    print(f"   📊 Columnas: {list(df.columns)}")
    print(f"   🔍 Ejemplo: {df.iloc[0].to_dict() if len(df) > 0 else 'Vacío'}\n")
print(f"✅ Todo en RAM. Sin escritura en disco. Listo para esqueleto + joins.")
print(SEP)
print(f"✅ Etapa 8 completada. ")

2026-06-24 10:40:49,881 - INFO - 📐 Construyendo 8 capas analíticas en RAM...
2026-06-24 10:40:49,892 - INFO - ✅ casos_vif_nna_f: 523 filas | columnas: ['cod_municipio', 'anio_hecho', 'cantidad']
2026-06-24 10:40:49,905 - INFO - ✅ casos_vif_adultas_f: 1,327 filas | columnas: ['cod_municipio', 'anio_hecho', 'cantidad']
2026-06-24 10:40:49,912 - INFO - ✅ casos_sexual_nna_f: 1,024 filas | columnas: ['cod_municipio', 'anio_hecho', 'cantidad']
2026-06-24 10:40:49,920 - INFO - ✅ casos_sexual_adultas_f: 1,272 filas | columnas: ['cod_municipio', 'anio_hecho', 'cantidad']
2026-06-24 10:40:49,927 - INFO - ✅ casos_vif_nna_m: 372 filas | columnas: ['cod_municipio', 'anio_hecho', 'cantidad']
2026-06-24 10:40:49,934 - INFO - ✅ casos_vif_adultos_m: 1,021 filas | columnas: ['cod_municipio', 'anio_hecho', 'cantidad']
2026-06-24 10:40:49,940 - INFO - ✅ casos_sexual_nna_m: 416 filas | columnas: ['cod_municipio', 'anio_hecho', 'cantidad']
2026-06-24 10:40:49,945 - INFO - ✅ casos_sexual_adultos_m: 608 filas

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload

✅ CELDA 6 - DATA AGGREGATOR EJECUTADO
🔹 casos_vif_nna_f: 523 filas
   📊 Columnas: ['cod_municipio', 'anio_hecho', 'cantidad']
   🔍 Ejemplo: {'cod_municipio': '19001', 'anio_hecho': 2018, 'cantidad': 47}

🔹 casos_vif_adultas_f: 1,327 filas
   📊 Columnas: ['cod_municipio', 'anio_hecho', 'cantidad']
   🔍 Ejemplo: {'cod_municipio': '19001', 'anio_hecho': 2018, 'cantidad': 1181}

🔹 casos_sexual_nna_f: 1,024 filas
   📊 Columnas: ['cod_municipio', 'anio_hecho', 'cantidad']
   🔍 Ejemplo: {'cod_municipio': '19001', 'anio_hecho': 2018, 'cantidad': 118}

🔹 casos_sexual_adultas_f: 1,272 filas
   📊 Columnas: ['cod_municipio', 'anio_hecho', 'cantidad']
   🔍 Ejemplo: {'cod_municipio': '19001', 'anio_hecho': 2018, 'cantidad': 215}

🔹 casos_vif_nna_m: 372 filas
   📊 Columnas: ['cod_municipio', 'anio_hecho', 'cantidad']
   🔍 Ejemplo: {'cod_municipio': '19001', 'anio_hecho': 2018, 'cantidad': 27}

🔹 casos_vif_adultos

In [9]:
import sys, json, pandas as pd
from pathlib import Path
%load_ext autoreload
%autoreload 2

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "config").exists() else Path.cwd().parent
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path: sys.path.insert(0, str(SRC_DIR))

# ✅ 1. Verificación de dependencia de memoria (viene de la celda de agregación)
if 'layers_ram' not in globals():
    raise RuntimeError("⚠️ 'layers_ram' no encontrado en RAM. Ejecuta primero la celda del DataAggregator en este mismo kernel.")

# ✅ 2. Carga temporal de limpios para comparación (diccionario en RAM, NO carpeta)
CLEAN_DIR = (PROJECT_ROOT / "data/cleaned").resolve()
cleaned_datasets = {
    "violencia": pd.read_parquet(CLEAN_DIR / "violencia_limpio.parquet"),
    "sexuales": pd.read_parquet(CLEAN_DIR / "sexuales_limpio.parquet")
}

# ✅ 3. Carga de config para formato (100% desde JSON)
base_path = PROJECT_ROOT / "config" / "base_config.json"
base_cfg = {}
if base_path.exists():
    with open(base_path, 'r', encoding='utf-8') as f: base_cfg = json.load(f)
with open(PROJECT_ROOT / "config" / "master_builder_config.json", 'r', encoding='utf-8') as f:
    spec_cfg = json.load(f)
config = {**base_cfg, **spec_cfg}

disp = config.get('display_config', {})
SEP = disp.get('separator_char', '=') * disp.get('separator_length', 70)
IND = disp.get('indent', '   ')
PREFIX = disp.get('prefix_dataset', '📦 ')
TITLE = disp.get('section_title', '✅ CELDA {n} - {module} EJECUTADO').format(n=8, module="VALIDACIÓN CAPAS")

from validator import LayerValidator

print(f"\n{TITLE}")
print(SEP)

validator = LayerValidator(config_path=str(PROJECT_ROOT / "config" / "master_builder_config.json"))
report = validator.validate(layers_ram, cleaned_datasets)

print(f"\n{IND}📊 Resultados:")
for check in report['checks']:
    icon = "✅" if check['status'] == 'PASS' else "❌"
    print(f"{IND}{PREFIX}{icon} {check['check']:<25} | {check.get('group', check.get('layer', '')):<15} | {check['msg']}")

print(SEP)
print(f"{IND}🛡️ Estado global: {report['status']}")
print(f"{IND}📄 Reporte trazable: docs/{validator.output_filename}")
print(f"{IND}✅ Capas validadas. Listo para Esqueleto Territorial + MasterBuilder.")
print(SEP)
print(f"✅ Etapa 9 completada.")

2026-06-24 10:40:53,632 - INFO - 📄 Reporte exportado: /Users/anaaguirre/Documents/Cicatrices_invisibles/docs/validacion_capas_pre_join.csv
2026-06-24 10:40:53,632 - INFO - ✅ Validación cruzada de capas: PASS. Pipeline continuando...


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload

✅ CELDA 8 - VALIDACIÓN CAPAS EJECUTADO

   📊 Resultados:
   📦 ✅ SUMA_TOTALES              | violencia_total | Suma violencia_total: 122825 vs 122825 (diff: 0.00%)
   📦 ✅ SUMA_TOTALES              | sexuales_total  | Suma sexuales_total: 34446 vs 34446 (diff: 0.00%)
   📦 ✅ UNICIDAD                  | casos_vif_nna_f | casos_vif_nna_f: Unicidad garantizada
   📦 ✅ COBERTURA_TEMPORAL        | casos_vif_nna_f | casos_vif_nna_f: Cobertura completa (2018-2025)
   📦 ✅ UNICIDAD                  | casos_vif_adultas_f | casos_vif_adultas_f: Unicidad garantizada
   📦 ✅ COBERTURA_TEMPORAL        | casos_vif_adultas_f | casos_vif_adultas_f: Cobertura completa (2018-2025)
   📦 ✅ UNICIDAD                  | casos_sexual_nna_f | casos_sexual_nna_f: Unicidad garantizada
   📦 ✅ COBERTURA_TEMPORAL        | casos_sexual_nna_f | casos_sexual_nna_f: Cobertura completa (2018-2025)
   📦 ✅ UNICIDAD                  | casos_

In [10]:
import pandas as pd
import json
from pathlib import Path

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "config").exists() else Path.cwd().parent

# ============================================================
# 🔧 GENERADOR DE JSON DESDE DANE (fuente única de verdad)
# Para uso con mapas CHOROPLETH (no requiere coordenadas)
# ============================================================

# 1. Cargar dataset de población limpio
pob_path = PROJECT_ROOT / "data" / "cleaned" / "poblacion_limpio.parquet"
pob_df = pd.read_parquet(pob_path)

# 2. Filtrar SOLO municipios del Pacífico (4 departamentos)
deptos_pacifico = ['CHOCO', 'VALLE DEL CAUCA', 'CAUCA', 'NARIÑO']
pob_pacifico = pob_df[pob_df['departamento'].str.upper().str.strip().isin(deptos_pacifico)]

# 3. Extraer municipios únicos
municipios_unicos = pob_pacifico[['cod_municipio', 'municipio', 'departamento']].drop_duplicates()
municipios_unicos = municipios_unicos.sort_values('cod_municipio')

print(f"📊 Municipios del Pacífico en DANE limpio: {len(municipios_unicos)}")
print(f"   • Departamentos: {sorted(pob_pacifico['departamento'].unique().tolist())}")

# 4. Construir JSON nuevo desde cero (solo nombre + departamento)
municipios_nuevo = {}

for _, row in municipios_unicos.iterrows():
    cod = str(row['cod_municipio']).strip()
    nombre = str(row['municipio']).strip().upper()
    depto = str(row['departamento']).strip().upper()

    municipios_nuevo[cod] = {
        "nombre": nombre,
        "departamento": depto
    }

# 5. Estructura final del JSON
json_nuevo = {
    "metadata": {
        "region": "PACIFICO_COLOMBIANO",
        "source": "DANE_DIVIPOLA",
        "version": "2024",
        "description": "Referencia geográfica para mapas coropléticos (une por cod_municipio con GeoJSON)",
        "uso": "choropleth",
        "total_municipios": len(municipios_nuevo),
        "departamentos": sorted(pob_pacifico['departamento'].unique().tolist()),
        "last_updated": pd.Timestamp.now().isoformat()
    },
    "municipios": municipios_nuevo
}

# 6. Guardar JSON nuevo (sobrescribe el viejo)
json_path = PROJECT_ROOT / "config" / "municipios_pacifico.json"
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(json_nuevo, f, indent=2, ensure_ascii=False)

print(f"\n✅ JSON generado desde DANE:")
print(f"   • Total municipios: {len(municipios_nuevo)}")
print(f"   • Tipo de mapa: choropleth (sin coordenadas)")
print(f"   • Archivo: {json_path}")
print(f"\n📋 Distribución por departamento:")
for depto in sorted(pob_pacifico['departamento'].unique()):
    n = len(municipios_unicos[municipios_unicos['departamento'] == depto])
    print(f"   • {depto}: {n} municipios")
print(f"✅ Etapa 9.5 completada. JSON regenerado desde DANE")

📊 Municipios del Pacífico en DANE limpio: 179
   • Departamentos: ['CAUCA', 'CHOCO', 'NARIÑO', 'VALLE DEL CAUCA']

✅ JSON generado desde DANE:
   • Total municipios: 179
   • Tipo de mapa: choropleth (sin coordenadas)
   • Archivo: /Users/anaaguirre/Documents/Cicatrices_invisibles/config/municipios_pacifico.json

📋 Distribución por departamento:
   • CAUCA: 42 municipios
   • CHOCO: 31 municipios
   • NARIÑO: 64 municipios
   • VALLE DEL CAUCA: 42 municipios
✅ Etapa 9.5 completada. JSON regenerado desde DANE


In [11]:
import sys, json, pandas as pd
from pathlib import Path
%load_ext autoreload
%autoreload 2

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "config").exists() else Path.cwd().parent
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path: sys.path.insert(0, str(SRC_DIR))

if 'layers_ram' not in globals():
    raise RuntimeError("⚠️ 'layers_ram' no encontrado. Ejecuta agregación y validación primero.")

CLEAN_DIR = (PROJECT_ROOT / "data/cleaned").resolve()
dane_df = pd.read_parquet(CLEAN_DIR / "poblacion_limpio.parquet")

base_path = PROJECT_ROOT / "config" / "base_config.json"
base_cfg = {}
if base_path.exists():
    with open(base_path, 'r', encoding='utf-8') as f: base_cfg = json.load(f)
with open(PROJECT_ROOT / "config" / "master_builder_config.json", 'r', encoding='utf-8') as f:
    spec_cfg = json.load(f)
config = {**base_cfg, **spec_cfg}

disp = config.get('display_config', {})
SEP = disp.get('separator_char', '=') * disp.get('separator_length', 70)
IND = disp.get('indent', '   ')
PREFIX = disp.get('prefix_dataset', '📦 ')
TITLE = disp.get('section_title', '✅ CELDA {n} - {module} EJECUTADO').format(n=9, module="DATA MASTER BUILDER")

from masterbuilder import DataMasterBuilder

print(f"\n{TITLE}")
print(SEP)

builder = DataMasterBuilder(config_path=str(PROJECT_ROOT / "config" / "master_builder_config.json"))
master_df = builder.build(layers_ram, dane_df)

print(f"\n{IND}📊 Resumen Tabla Maestra:")
print(f"{IND}{PREFIX} Filas maestro anual: {len(master_df):,} | Columnas: {len(master_df.columns)}")
print(f"{IND}{PREFIX} Filas tabla clustering: {pd.read_parquet(builder.master_dir / builder.output_cfg['clustering_filename']).shape[0]}")
print(f"{IND}{PREFIX} Rango ICV-GEN-F: {master_df['icv_gen_f_score'].min():.2f} - {master_df['icv_gen_f_score'].max():.2f}")
print(SEP)
print(f"{IND}🛡️ Garantías aplicadas:")
print(f"{IND}• Esqueleto DANE × años (NaN en casos → 0 | NaN en población → preservado)")
print(f"{IND}• Brechas f/m con manejo seguro de división por cero → NaN")
print(f"{IND}• ICV-GEN-F normalizado Min-Max y ponderado por JSON")
print(f"{IND}• Tabla de clustering colapsada por municipio para K-Means")
print(SEP)
print(f"✅ Etapa 10 completada. Pipeline cerrado. Listo para EDA + ML.")

2026-06-24 10:41:03,169 - INFO - 🏗️ Construyendo Tabla Maestra (Esqueleto + JOINs + Tasas + Brechas + ICV)...
2026-06-24 10:41:03,174 - INFO - 🔍 Población filtrada por area_geo='Total': 4,296 → 1,432 filas (factor 3.0x)
2026-06-24 10:41:03,206 - INFO - 💾 Maestro anual exportado: /Users/anaaguirre/Documents/Cicatrices_invisibles/data/master/master_table.parquet (1,432 filas)
2026-06-24 10:41:03,211 - INFO - 💾 Tabla Clustering exportada: /Users/anaaguirre/Documents/Cicatrices_invisibles/data/master/tabla_clustering.parquet (179 filas)


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload

✅ CELDA 9 - DATA MASTER BUILDER EJECUTADO

   📊 Resumen Tabla Maestra:
   📦  Filas maestro anual: 1,432 | Columnas: 31
   📦  Filas tabla clustering: 179
   📦  Rango ICV-GEN-F: 0.00 - 65.54
   🛡️ Garantías aplicadas:
   • Esqueleto DANE × años (NaN en casos → 0 | NaN en población → preservado)
   • Brechas f/m con manejo seguro de división por cero → NaN
   • ICV-GEN-F normalizado Min-Max y ponderado por JSON
   • Tabla de clustering colapsada por municipio para K-Means
✅ Etapa 10 completada. Pipeline cerrado. Listo para EDA + ML.


###**Revision**

In [12]:
import pandas as pd
from pathlib import Path

# 🔒 Misma lógica de resolución de rutas del pipeline
PROJECT_ROOT = Path.cwd() if (Path.cwd() / "config").exists() else Path.cwd().parent
master_path = PROJECT_ROOT / "data" / "master" / "master_table.parquet"

if not master_path.exists():
    raise FileNotFoundError(
        f"❌ No se encontró {master_path}.\n"
        f"¿Ya ejecutaste la celda 10 del ETL con el MasterBuilder corregido?"
    )

master = pd.read_parquet(master_path)

# 1. Cardinalidad esperada
n_mun = master['cod_municipio'].nunique()
n_anios = master['anio_hecho'].nunique()
esperado = n_mun * n_anios
actual = len(master)

print(f"📂 Ruta leída: {master_path}")
print(f"📊 Municipios únicos: {n_mun}")
print(f"📊 Años únicos: {n_anios}")
print(f"📊 Filas esperadas (1:1): {esperado:,}")
print(f"📊 Filas actuales: {actual:,}")
print(f"{'✅ SIN DUPLICADOS' if actual == esperado else '❌ AÚN HAY DUPLICADOS'}")

# 2. Detección de duplicados residuales
dup = master.groupby(['cod_municipio', 'anio_hecho']).size()
print(f"\n🔎 Max filas por (municipio, año): {dup.max()}")
print(f"🔎 Combinaciones con >1 fila: {(dup > 1).sum()}")

# 3. Verificación del filtro de población
print(f"\n🔎 Columnas en master: {list(master.columns)}")
if 'area_geo' in master.columns:
    print(f"⚠️ 'area_geo' aún está en master (valores únicos): {master['area_geo'].unique()}")
else:
    print(f"✅ 'area_geo' no está en master (esperado, se filtró antes del merge)")

📂 Ruta leída: /Users/anaaguirre/Documents/Cicatrices_invisibles/data/master/master_table.parquet
📊 Municipios únicos: 179
📊 Años únicos: 8
📊 Filas esperadas (1:1): 1,432
📊 Filas actuales: 1,432
✅ SIN DUPLICADOS

🔎 Max filas por (municipio, año): 1
🔎 Combinaciones con >1 fila: 0

🔎 Columnas en master: ['cod_municipio', 'anio_hecho', 'municipio', 'departamento', 'pob_f_0_17', 'pob_f_18_mas', 'pob_h_0_17', 'pob_h_18_mas', 'total_f', 'total_h', 'casos_vif_nna_f', 'casos_vif_adultas_f', 'casos_sexual_nna_f', 'casos_sexual_adultas_f', 'casos_vif_nna_m', 'casos_vif_adultos_m', 'casos_sexual_nna_m', 'casos_sexual_adultos_m', 'tasa_vif_nna_f', 'tasa_vif_adultas_f', 'tasa_sexual_nna_f', 'tasa_sexual_adultas_f', 'tasa_vif_nna_m', 'tasa_vif_adultos_m', 'tasa_sexual_nna_m', 'tasa_sexual_adultos_m', 'brecha_vif_nna', 'brecha_vif_adultas', 'brecha_sexual_nna', 'brecha_sexual_adultas', 'icv_gen_f_score']
✅ 'area_geo' no está en master (esperado, se filtró antes del merge)


In [13]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "config").exists() else Path.cwd().parent

# Cargar tabla de clustering
clustering = pd.read_parquet(PROJECT_ROOT / "data/master/tabla_clustering.parquet")

# Verificar NaN en nombres
nan_count = clustering['municipio'].isna().sum()
nan_munis = clustering[clustering['municipio'].isna()]

print(f"📊 Verificación de nombres de municipios:")
print(f"   • Total municipios: {len(clustering)}")
print(f"   • Con nombre válido: {len(clustering) - nan_count}")
print(f"   • Con nombre NaN: {nan_count}")

if nan_count > 0:
    print(f"\n⚠️ Municipios sin nombre:")
    print(nan_munis[['cod_municipio', 'departamento']].to_string(index=False))
else:
    print(f"\n✅ Todos los municipios tienen nombre válido")
    print(f"\n📋 Muestra de municipios:")
    print(clustering[['cod_municipio', 'municipio', 'departamento']].head(10).to_string(index=False))

📊 Verificación de nombres de municipios:
   • Total municipios: 179
   • Con nombre válido: 179
   • Con nombre NaN: 0

✅ Todos los municipios tienen nombre válido

📋 Muestra de municipios:
cod_municipio    municipio departamento
        19001      POPAYAN        CAUCA
        19022     ALMAGUER        CAUCA
        19050      ARGELIA        CAUCA
        19075       BALBOA        CAUCA
        19100      BOLIVAR        CAUCA
        19110 BUENOS AIRES        CAUCA
        19130      CAJIBIO        CAUCA
        19137      CALDONO        CAUCA
        19142       CALOTO        CAUCA
        19212      CORINTO        CAUCA


In [14]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "config").exists() else Path.cwd().parent
master_path = PROJECT_ROOT / "data" / "master" / "master_table.parquet"

df = pd.read_parquet(master_path)

print("📊 Estadísticas de brechas de género:")
print("=" * 70)

brechas = ['brecha_vif_nna', 'brecha_vif_adultas', 'brecha_sexual_nna', 'brecha_sexual_adultas']

for b in brechas:
    validos = df[b].notna().sum()
    porcentaje = validos / len(df) * 100
    media = df[b].mean()
    mediana = df[b].median()

    print(f"\n🔍 {b}:")
    print(f"   • Valores válidos: {validos} de {len(df)} ({porcentaje:.1f}%)")
    print(f"   • Media: {media:.2f}")
    print(f"   • Mediana: {mediana:.2f}")
    print(f"   • Interpretación: {'Mayor afectación femenina' if media > 1 else 'Mayor afectación masculina' if media < 1 else 'Paridad'}")

📊 Estadísticas de brechas de género:

🔍 brecha_vif_nna:
   • Valores válidos: 372 de 1432 (26.0%)
   • Media: 1.67
   • Mediana: 1.34
   • Interpretación: Mayor afectación femenina

🔍 brecha_vif_adultas:
   • Valores válidos: 1019 de 1432 (71.2%)
   • Media: 4.97
   • Mediana: 3.89
   • Interpretación: Mayor afectación femenina

🔍 brecha_sexual_nna:
   • Valores válidos: 416 de 1432 (29.1%)
   • Media: 4.86
   • Mediana: 4.29
   • Interpretación: Mayor afectación femenina

🔍 brecha_sexual_adultas:
   • Valores válidos: 608 de 1432 (42.5%)
   • Media: 5.08
   • Mediana: 4.29
   • Interpretación: Mayor afectación femenina
